# Trích xuất đặc trưng cho dataset C++ (Content Database)

Notebook trích xuất đặc trưng theo tài liệu *Content_Database_v2_CodeOnly_Full.pdf* cho
dataset câu hỏi C++ dạng **output prediction / code tracing**.

- **Phase 2 — Surface features** (regex/AST, offline): #1–#14, #20–#21.
- **Phase 3 — LLM features** (gọi server LLM): #15–#23.
- Bỏ Phase 1 (embedding 768d). Output: `features.csv` + `features.json`.

Nguồn: `../prepare_dataset/questions_db_ready.json` (1393 câu).
Chạy bằng kernel **uv** của project. Thực hiện **từng cell một**.

## Bước 0 — Setup & load dữ liệu

In [1]:
import re
import json
import math
import statistics
import difflib
from pathlib import Path
from collections import Counter

import pandas as pd
import requests

# ----- Đường dẫn IO -----
HERE = Path.cwd()
DATA_PATH = (HERE / ".." / "prepare_dataset" / "questions_db_ready.json").resolve()
OUT_DIR = HERE.resolve()
print("DATA_PATH:", DATA_PATH, "| tồn tại:", DATA_PATH.exists())
print("OUT_DIR  :", OUT_DIR)

with open(DATA_PATH, encoding="utf-8") as f:
    raw = json.load(f)

df = pd.DataFrame(raw)
print("Số câu hỏi:", len(df))
df.head(3)

DATA_PATH: D:\IntelligentTesting\intelligent-testing\notebooks\prepare_dataset\questions_db_ready.json | tồn tại: True
OUT_DIR  : D:\IntelligentTesting\intelligent-testing\notebooks\extract_feature
Số câu hỏi: 1393


,question_id,question_content,all_option_ids,all_options_content,correct_option_ids,skill_ids
0,7,Kết quả thực hiện đoạn chương trình sau là gì?...,"[30, 31, 32, 33, 34]","[12, 22, 32, 23, 33]",[31],[1]
1,8,Kết quả thực thi đoạn chương trình sau là gì? ...,"[35, 36, 37, 38, 39]","[12, 22, 32, 23, 33]",[36],[1]
2,9,Kết quả thực hiện đoạn chương trình sau là gì?...,"[40, 41, 42, 43, 44, 1225]","[Compile Error, 1, 2, 3, Runtime Error, Anothe...",[42],[1]


## Bước 1 — Tách code C++ khỏi câu hỏi tiếng Việt

`question_content` lẫn cả phần mô tả tiếng Việt và code. Heuristic: cắt tại `#include`
đầu tiên; nếu không có `#include` thì dò các dấu hiệu code (`int main`, `cout`, `{`...);
nếu không có code → `code=""` (câu lý thuyết).

In [2]:
CODE_HINTS = ("int main", "void main", "cout", "cin", "using namespace",
              "printf", "scanf", "return 0", "#define")

def split_question(content: str):
    """Trả về (question_text, code)."""
    content = content or ""
    idx = content.find("#include")
    if idx != -1:
        return content[:idx].strip(), content[idx:].strip()
    # Không có #include: dò dấu hiệu code đầu tiên
    positions = [content.find(h) for h in CODE_HINTS if content.find(h) != -1]
    if positions:
        cut = min(positions)
        # lùi về đầu dòng chứa dấu hiệu để giữ trọn code
        nl = content.rfind("\n", 0, cut)
        cut = nl + 1 if nl != -1 else cut
        return content[:cut].strip(), content[cut:].strip()
    return content.strip(), ""  # câu lý thuyết, không có code

df[["question_text", "code"]] = df["question_content"].apply(
    lambda c: pd.Series(split_question(c))
)

n_has_code = (df["code"].str.len() > 0).sum()
print(f"Câu có code: {n_has_code}/{len(df)}")
for qid in (7, 9, 10):
    row = df[df.question_id == qid].iloc[0]
    print(f"\n===== Q{qid} =====\n[text] {row.question_text!r}\n[code]\n{row.code}")

Câu có code: 1239/1393

===== Q7 =====
[text] 'Kết quả thực hiện đoạn chương trình sau là gì?'
[code]
#include using namespace std;
int main()
{
	int var_x = 2, & ref_x = var_x;
	cout << (++var_x) << (ref_x--);
	return 0;
}

===== Q9 =====
[text] 'Kết quả thực hiện đoạn chương trình sau là gì?'
[code]
#include using namespace std;
int main()
{
	enum Exam
	{
		Exam1 = 1,
		Exam2,
		Exam3
	};
	int var_x = Exam3;
	int& ref_x = var_x;
	int& ref_y = var_x;
	ref_x = Exam2;
	cout << ref_y--;
	return 0;
}

===== Q10 =====
[text] 'Kết quả thực thi đoạn chương trình sau là gì?'
[code]
#include using namespace std;
int main()
{
	int var_x = 0;
	int& ref_x = var_x;
	ref_x = 4;
	while (var_x <= 4)
	{
		cout << ref_x++;
		var_x++;
	}
	cout << var_x;
	return 0;
}


## Bước 2 — Surface lexical (mục 1A)

`L_qtok` (token câu hỏi), `L_lines` (số dòng code), `L_kw` (mật độ từ khóa C++),
`L_ids` (số identifier do người dùng định nghĩa).

In [3]:
# Từ khóa C++ 3 tầng (theo PDF)
KW_T1 = {"int","float","double","char","bool","void","if","else","for","while",
         "do","return","cout","cin"}
KW_T2 = {"class","struct","enum","namespace","const","static"}
KW_T3 = {"virtual","override","template","friend","new","delete","dynamic_cast","typeid"}
CPP_KEYWORDS = KW_T1 | KW_T2 | KW_T3
# STL/identifier chuẩn cần loại khi đếm identifier người dùng
STL_NAMES = {"std","cout","cin","endl","string","vector","map","set","pair","main",
             "printf","scanf","include","iostream","using","namespace","system",
             "size_t","nullptr","NULL","true","false","this"}

WORD_RE = re.compile(r"[A-Za-z_]\w*")

def code_tokens(code: str):
    return WORD_RE.findall(code)

def f_qtok(text: str) -> int:
    return len(text.split())

def f_lines(code: str) -> int:
    return len(code.strip().split("\n")) if code.strip() else 0

def f_kw_density(code: str) -> float:
    toks = code_tokens(code)
    if not toks:
        return 0.0
    return sum(1 for t in toks if t in CPP_KEYWORDS) / len(toks)

def f_user_ids(code: str) -> int:
    toks = code_tokens(code)
    ids = {t for t in toks if t not in CPP_KEYWORDS and t not in STL_NAMES}
    return len(ids)

df["L_qtok"]  = df["question_text"].map(f_qtok)
df["L_lines"] = df["code"].map(f_lines)
df["L_kw"]    = df["code"].map(f_kw_density)
df["L_ids"]   = df["code"].map(f_user_ids)
df[["question_id","L_qtok","L_lines","L_kw","L_ids"]].head()

,question_id,L_qtok,L_lines,L_kw,L_ids
0,7,10,7,0.357143,2
1,8,10,9,0.352941,3
2,9,10,16,0.320000,7
3,10,10,14,0.400000,2
4,11,10,11,0.259259,4


## Bước 3 — Surface syntactic (mục 1B)

`S_nest` (độ sâu lồng ngoặc nhọn), `S_cf` (control-flow score),
`S_ops[6]` (vector tỷ lệ 6 nhóm toán tử đặc thù).

In [4]:
def f_max_nesting(code: str) -> int:
    depth = max_depth = 0
    for ch in code:
        if ch == "{":
            depth += 1
            max_depth = max(max_depth, depth)
        elif ch == "}":
            depth -= 1
    return max_depth

def _wcount(pattern, code):
    return len(re.findall(pattern, code))

def f_control_flow(code: str) -> float:
    n_if     = _wcount(r"\b(if|else)\b", code)
    n_for    = _wcount(r"\bfor\b", code)
    n_while  = _wcount(r"\b(while|do)\b", code)
    n_switch = _wcount(r"\bswitch\b", code)
    return n_if + 1.5 * n_for + 1.5 * n_while + 1.2 * n_switch

# Từ khóa kiểu — để phân biệt con trỏ/reference THẬT với phép nhân (a*b) / bitwise (a&b)
_TYPE = r"(?:int|long|short|unsigned|char|float|double|bool|void|auto|wchar_t|size_t|string|[A-Z]\w*)"

def _count_ptr_star(code: str) -> int:
    """Đếm '*' là con trỏ/deref, KHÔNG tính phép nhân a*b.
    - khai báo/ép kiểu:  type *   (int *p, Node* x)
    - deref:  *ident  đứng sau toán tử/dấu mở/đầu dòng  (= *p, (*p), << *p)"""
    decl  = re.findall(_TYPE + r"\s*\*", code)
    deref = re.findall(r"(?:^|[(,={;\[+\-*/%<>&|!?:])\s*\*\s*[A-Za-z_]", code, re.M)
    return len(decl) + len(deref)

def _count_ref_amp(code: str) -> int:
    """Đếm '&' là reference/address, loại '&&' (logic) và bitwise a&b.
    - khai báo reference:  type &   (int& r, Node &x)
    - binding/address-of:  & ident  đứng sau toán tử/dấu mở/dấu phẩy/đầu dòng"""
    decl = re.findall(_TYPE + r"\s*&(?!&)", code)
    addr = re.findall(r"(?:^|[(,={;\[+\-*/%<>|!?:])\s*&(?!&)\s*[A-Za-z_]", code, re.M)
    return len(decl) + len(addr)

# 4 toán tử còn lại tính trực tiếp; d_ptr & d_ref dùng hàm ở trên
OP_SIMPLE = [
    ("d_arrow",  r"->"),                # member ptr
    ("d_scope",  r"::"),                # scope
    ("d_incr",   r"\+\+|--"),           # ++ / --
    ("d_assign", r"(?<![=!<>])=(?!=)"),  # gán = (không phải ==, !=, <=, >=)
]

def f_operator_vector(code: str):
    # Vector: [d_ptr, d_ref, d_arrow, d_scope, d_incr, d_assign] (tỷ lệ / tổng token)
    toks = code_tokens(code)
    denom = len(toks) if toks else 1
    rest = [len(re.findall(p, code)) / denom for _n, p in OP_SIMPLE]
    return [_count_ptr_star(code) / denom, _count_ref_amp(code) / denom] + rest

df["S_nest"] = df["code"].map(f_max_nesting)
df["S_cf"]   = df["code"].map(f_control_flow)
df["S_ops"]  = df["code"].map(f_operator_vector)
df[["question_id","S_nest","S_cf","S_ops"]].head()

,question_id,S_nest,S_cf,S_ops
0,7,1,0.0,"[0.0, 0.07142857142857142, 0.0, 0.0, 0.1428571..."
1,8,1,0.0,"[0.0, 0.058823529411764705, 0.0, 0.0, 0.117647..."
2,9,2,0.0,"[0.0, 0.08, 0.0, 0.0, 0.04, 0.2]"
3,10,2,1.5,"[0.0, 0.05, 0.0, 0.0, 0.1, 0.15]"
4,11,1,0.0,"[0.0, 0.07407407407407407, 0.0, 0.0, 0.1481481..."


## Bước 4 — Surface structural (mục 1C)

`T_class`, `T_oop[3]=[N_class,D_inherit,R_count]`,
`T_mem[4]=[is_stack,is_heap,is_static,is_global]`, `T_type` (mức 1–5).

In [5]:
CLASS_RE = re.compile(r"\b(class|struct)\s+\w+")

def f_class_count(code: str) -> int:
    return len(CLASS_RE.findall(code))

def f_oop_vector(code: str):
    n_class = f_class_count(code)
    # Độ sâu kế thừa: mỗi khai báo ": public/private/protected Base"
    inherit_edges = re.findall(r":\s*(?:public|private|protected)\s+\w+", code)
    d_inherit = 1 if inherit_edges else 0   # heuristic: có kế thừa => sâu >=1
    # Quan hệ tổng: kế thừa + composition (class này chứa biến kiểu class khác) ~ xấp xỉ
    r_count = len(inherit_edges)
    return [n_class, d_inherit, r_count]

def f_memory_vector(code: str):
    is_heap   = 1 if re.search(r"\b(new|malloc|calloc)\b", code) else 0
    is_static = 1 if re.search(r"\bstatic\b", code) else 0
    # global: BIẾN (không phải hàm) khai báo trước 'int main'
    head = code.split("int main")[0] if "int main" in code else ""
    is_global = 0
    for line in head.splitlines():
        if not re.match(r"\s*(?:const\s+|static\s+)*"
                        r"(?:int|long|short|unsigned|char|float|double|bool)\b", line):
            continue
        # bỏ qua dòng là HÀM/prototype ('('), preprocessor ('#'), using/typedef
        ls = line.lstrip()
        if "(" in line or ls.startswith("#") or "using" in line or "typedef" in line:
            continue
        if "=" in line or line.rstrip().endswith(";"):   # đúng là khai báo biến
            is_global = 1
            break
    is_stack  = 1 if code.strip() else 0    # có code => có biến stack
    return [is_stack, is_heap, is_static, is_global]

def f_type_complexity(code: str) -> int:
    if re.search(r"<\s*\w+.*?>|shared_ptr|unique_ptr|weak_ptr", code):
        return 5  # template / smart pointer
    if re.search(r"\*\s*\*|" + _TYPE + r"\s*\*\s*\[|\[\s*\]\s*\*", code):
        return 4  # pointer-to-pointer / array of pointer
    # con trỏ / reference THẬT (dùng lại bộ đếm ở Bước 3 — đã loại a*b, a&b)
    if (_count_ptr_star(code) or _count_ref_amp(code)
            or re.search(r"->|\bnew\b|\bdelete\b|\bmalloc\b", code)):
        return 3
    if re.search(r"\[\s*\d*\s*\]|\b(struct|enum)\s+\w+", code):
        return 2  # mảng / struct / enum
    if re.search(r"\b(int|float|double|char|bool|long|short|unsigned)\b", code):
        return 1  # primitive
    return 0

df["T_class"] = df["code"].map(f_class_count)
df["T_oop"]   = df["code"].map(f_oop_vector)
df["T_mem"]   = df["code"].map(f_memory_vector)
df["T_type"]  = df["code"].map(f_type_complexity)
df[["question_id","T_class","T_oop","T_mem","T_type"]].head()

,question_id,T_class,T_oop,T_mem,T_type
0,7,0,"[0, 0, 0]","[1, 0, 0, 0]",3
1,8,0,"[0, 0, 0]","[1, 0, 0, 0]",3
2,9,0,"[0, 0, 0]","[1, 0, 0, 0]",3
3,10,0,"[0, 0, 0]","[1, 0, 0, 0]",3
4,11,0,"[0, 0, 0]","[1, 0, 0, 0]",3


## Bước 5 — Option features (mục 1D)

`O_var` (variance độ dài option), `O_spc[3]=[CompileError,RuntimeError,AnotherAnswer]`,
`O_sim` (độ tương đồng giữa các option thuần số).

In [7]:
def f_option_len_var(options):
    lens = [len(str(o)) for o in options]
    return statistics.pvariance(lens) if len(lens) > 1 else 0.0

def f_special_flags(options):
    low = [str(o).strip().lower() for o in options]
    return [
        int(any("compile error" in o for o in low)),
        int(any("runtime error" in o for o in low)),
        int(any("another answer" in o for o in low)),
    ]

def _is_numeric(o):
    return str(o).strip().lstrip("-").replace(" ", "").isdigit()

def f_numeric_similarity(options):
    nums = [str(o).strip() for o in options if _is_numeric(o)]
    if len(nums) < 2:
        return 0.0
    sims = [
        difflib.SequenceMatcher(None, a, b).ratio()
        for i, a in enumerate(nums) for b in nums[i + 1:]
    ]
    return sum(sims) / len(sims)

df["O_var"] = df["all_options_content"].map(f_option_len_var)
df["O_spc"] = df["all_options_content"].map(f_special_flags)
df["O_sim"] = df["all_options_content"].map(f_numeric_similarity)
df[["question_id","O_var","O_spc","O_sim"]].head()

,question_id,O_var,O_spc,O_sim
0,7,0.000000,"[0, 0, 0]",0.400000
1,8,0.000000,"[0, 0, 0]",0.400000
2,9,38.138889,"[1, 1, 1]",0.000000
3,10,21.805556,"[1, 1, 1]",0.660173
4,11,23.040000,"[0, 0, 1]",0.000000


## Bước 6 — Sanity-check & lưu surface checkpoint

Đối chiếu câu #7 và #10 với ví dụ PHẦN 4 của PDF, rồi lưu `features_surface.{csv,json}`
để Phase 3 có thể chạy độc lập.

In [7]:
check_cols = ["question_id","L_lines","S_nest","S_cf","S_ops","T_type","O_sim"]
print("Đối chiếu PDF (PHẦN 4):")
print(df[df.question_id.isin([7, 10])][check_cols].to_string(index=False))
# Kỳ vọng: Q7 L_lines nhỏ, S_nest=0 ; Q10 S_nest=1, S_cf=1.5

def explode_vectors(frame):
    """Tách các cột vector thành nhiều cột số cho CSV."""
    out = frame.copy()
    vec_cols = {
        "S_ops": ["d_ptr","d_ref","d_arrow","d_scope","d_incr","d_assign"],
        "T_oop": ["oop_nclass","oop_dinherit","oop_rcount"],
        "T_mem": ["mem_stack","mem_heap","mem_static","mem_global"],
        "O_spc": ["spc_compile","spc_runtime","spc_another"],
    }
    for col, names in vec_cols.items():
        if col in out.columns:
            mat = pd.DataFrame(out[col].tolist(), index=out.index, columns=names)
            out = pd.concat([out.drop(columns=[col]), mat], axis=1)
    return out

SURFACE_COLS = ["question_id","L_qtok","L_lines","L_kw","L_ids","S_nest","S_cf",
                "S_ops","T_class","T_oop","T_mem","T_type","O_var","O_spc","O_sim"]
surface = df[SURFACE_COLS].copy()
surface.to_json(OUT_DIR / "features_surface.json", orient="records",
                force_ascii=False, indent=2)
explode_vectors(surface).to_csv(OUT_DIR / "features_surface.csv", index=False)
print("\nĐã lưu features_surface.csv / .json")

Đối chiếu PDF (PHẦN 4):
 question_id  L_lines  S_nest  S_cf                                                                          S_ops  T_type    O_sim
           7        7       1   0.0 [0.0, 0.07142857142857142, 0.0, 0.0, 0.14285714285714285, 0.14285714285714285]       3 0.400000
          10       14       2   1.5                                               [0.0, 0.05, 0.0, 0.0, 0.1, 0.15]       3 0.660173

Đã lưu features_surface.csv / .json


## Bước 6b — Phase 1: Code embedding bằng CodeBERT (E_code, 768-d)

Dùng `microsoft/codebert-base` (RoBERTa cho code, output 768 chiều) để nhúng đoạn code C++.
Vector mỗi câu = **mean pooling** của `last_hidden_state` theo attention mask.
Kết quả lưu riêng ra `code_embeddings.npy` (shape `[N, 768]`) + `code_embeddings_qid.npy`
để giữ thứ tự `question_id`.

> Lần đầu chạy sẽ **tải ~500MB** model. Trên CPU sẽ chậm (~15–30 phút cho 1393 câu);
> có GPU thì nhanh hơn nhiều. Embedding không phụ thuộc surface/LLM nên có thể chạy độc lập.

In [8]:
# ===== Phase 1 — Code embedding bằng CodeBERT (microsoft/codebert-base, 768-d = E_code) =====
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel, AutoConfig
from huggingface_hub import hf_hub_download

EMB_MODEL = "microsoft/codebert-base"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EMB_BATCH = 8 if DEVICE == "cpu" else 32     # CPU dùng batch nhỏ
EMB_MAXLEN = 512                              # giới hạn token của CodeBERT
print("Device:", DEVICE)

_tok = AutoTokenizer.from_pretrained(EMB_MODEL)
# codebert-base chỉ có pytorch_model.bin (không safetensors) -> transformers 5.x CHẶN
# from_pretrained khi torch < 2.6 (CVE-2025-32434). Lách an toàn: build từ config rồi
# load_state_dict (torch.load weights_only=True là an toàn). Nếu torch>=2.6 thì chạy thẳng.
try:
    _emb_model = AutoModel.from_pretrained(EMB_MODEL)
except Exception as e:
    print("from_pretrained bị chặn, chuyển sang load_state_dict thủ công:", type(e).__name__)
    _emb_model = AutoModel.from_config(AutoConfig.from_pretrained(EMB_MODEL))
    _sd = torch.load(hf_hub_download(EMB_MODEL, "pytorch_model.bin"),
                     map_location="cpu", weights_only=True)
    _miss, _unexp = _emb_model.load_state_dict(_sd, strict=False)
    assert not _unexp, f"unexpected keys: {list(_unexp)[:5]}"
_emb_model = _emb_model.to(DEVICE).eval()

@torch.no_grad()
def embed_codes(texts, batch=EMB_BATCH):
    vecs = []
    for i in range(0, len(texts), batch):
        chunk = [t if t.strip() else " " for t in texts[i:i + batch]]
        enc = _tok(chunk, padding=True, truncation=True, max_length=EMB_MAXLEN,
                   return_tensors="pt").to(DEVICE)
        out = _emb_model(**enc).last_hidden_state            # [B, L, 768]
        mask = enc["attention_mask"].unsqueeze(-1).float()
        # mean pooling theo attention mask (bỏ qua token padding)
        emb = (out * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        vecs.append(emb.cpu().numpy().astype("float32"))
        done = min(i + batch, len(texts))
        if (i // batch) % 10 == 0 or done == len(texts):
            print(f"  {done}/{len(texts)}")
    return np.vstack(vecs)

# Dùng code nếu có; câu lý thuyết (code rỗng) fallback sang nguyên văn câu hỏi
emb_input = [c if c.strip() else q for c, q in zip(df["code"], df["question_content"])]
E_code = embed_codes(emb_input)
print("Embedding shape:", E_code.shape)            # kỳ vọng (1393, 768)

np.save(OUT_DIR / "code_embeddings.npy", E_code)
np.save(OUT_DIR / "code_embeddings_qid.npy", df["question_id"].to_numpy())
df["E_code"] = list(E_code)                          # giữ trong df để gộp ở bước cuối
print("Đã lưu code_embeddings.npy", E_code.shape, "+ code_embeddings_qid.npy")

c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 53988.78it/s]


  8/1393
  88/1393
  168/1393
  248/1393
  328/1393
  408/1393
  488/1393
  568/1393
  648/1393
  728/1393
  808/1393
  888/1393
  968/1393
  1048/1393
  1128/1393
  1208/1393
  1288/1393
  1368/1393
  1393/1393
Embedding shape: (1393, 768)
Đã lưu code_embeddings.npy (1393, 768) + code_embeddings_qid.npy


## Bước 7 — LLM client + prompt hợp nhất (Phase 3)

Gọi thẳng server OpenAI-compatible bằng `requests`. Model là **reasoning model**
(trả `reasoning_content` riêng, đáp án nằm ở `content`) nên dùng `max_tokens` lớn và
parse JSON từ `content` (fallback regex). Một prompt hợp nhất/câu trả về đủ trường LLM
để giảm số lần gọi.

In [ ]:
# Hai server LLM (OpenAI-compatible) — gọi luân phiên để chạy song song, tăng tốc
LLM_SERVERS = [
    "https://llm.phuocnguyn.id.vn/v1",
    "https://llm2.dutai.site/v1",
]
LLM_MODEL = "unsloth/gemma-4-26B-A4B-it-GGUF"
LLM_HEADERS = {
    "Content-Type": "application/json",
    "Authorization": "Bearer sk-dummy",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Accept": "application/json",
}

_JSON_RE = re.compile(r"\{.*\}", re.S)

def _parse_json(text: str):
    if not text:
        return None
    t = text.strip()
    if t.startswith("```"):                       # bỏ code-fence ```json ... ```
        t = re.sub(r"^```[a-zA-Z]*\n?|\n?```$", "", t).strip()
    try:
        return json.loads(t)
    except Exception:
        m = _JSON_RE.search(t)
        if m:
            try:
                return json.loads(m.group(0))
            except Exception:
                return None
    return None

# Lưu ý: đây là reasoning model — tốn nhiều token cho phần suy luận (reasoning_content),
# đáp án JSON nằm ở 'content'. Cần max_tokens lớn, nếu không content sẽ rỗng (finish=length).
def call_llm(prompt: str, server: str = None, temperature: float = 0.1,
             max_tokens: int = 6144, retries: int = 2):
    server = server or LLM_SERVERS[0]
    payload = {
        "model": LLM_MODEL,
        "messages": [
            {"role": "system",
             "content": "Bạn là chuyên gia phân tích độ khó câu hỏi lập trình C++."},
            {"role": "user", "content": prompt},
        ],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    last_err = None
    for _ in range(retries + 1):
        try:
            r = requests.post(f"{server}/chat/completions", headers=LLM_HEADERS,
                              json=payload, timeout=120)
            r.raise_for_status()
            content = r.json()["choices"][0]["message"]["content"]
            parsed = _parse_json(content)
            if parsed is not None:
                return parsed
            last_err = "parse-fail"
        except Exception as e:
            last_err = str(e)
    return {"_error": last_err}

PROMPT_TEMPLATE = """Cho đoạn code C++ và các phương án trả lời sau.

CODE:
{code}

ĐÁP ÁN ĐÚNG: {correct}
CÁC PHƯƠNG ÁN: {options}

Hãy phân tích trong đầu (trace từng bước thực thi), sau đó CHỈ trả về một JSON duy nhất theo schema:
{{
  "step_count": <int, số bước trace nguyên tử>,
  "reasoning_depth": <int, độ sâu suy luận>,
  "weighted_steps": <number, tổng bước có trọng số: Identity*1.0, Retrieval*1.5, Simulation*1.0, Constraint*1.2>,
  "ambiguity": "<low|medium|high>",
  "bloom_level": <int 1-6: 1 Remember,2 Understand,3 Apply,4 Analyze,5 Evaluate,6 Create>,
  "misconceptions": [{{"type":"<Conceptual|Procedural|Side-effect trap|Syntactic trap|Lack of sense>","weight":<number>}}],
  "distractor_scores": [{{"option":"<nội dung phương án sai>","score":<int 1-5>,"trap_type":"<mô tả lỗi trace>"}}]
}}
Chỉ xuất JSON, không thêm chữ nào khác."""

def build_prompt(row):
    correct_ids = set(row["correct_option_ids"])
    ids = row["all_option_ids"]
    opts = row["all_options_content"]
    correct = ", ".join(str(o) for i, o in zip(ids, opts) if i in correct_ids)
    return PROMPT_TEMPLATE.format(
        code=row["code"] or row["question_content"],
        correct=correct,
        options=" | ".join(str(o) for o in opts),
    )

# Test thử: gọi MỖI server 1 lần để xác nhận cả 2 đều phản hồi & parse được
for _srv in LLM_SERVERS:
    _demo = call_llm(build_prompt(df[df.question_id == 10].iloc[0]),
                     server=_srv, temperature=0.0)
    print(_srv, "->", "OK" if isinstance(_demo, dict) and "_error" not in _demo else _demo)

## Bước 8 — Chạy Phase 3 có cache/resume

Mỗi câu gọi 3 temperature (0.0, 0.3, 0.7). Kết quả mỗi (question_id, temperature) ghi vào
`llm_cache.jsonl` để chạy lại không gọi trùng. Đặt `LIMIT` để chạy thử trước
(VD: `LIMIT=5`), rồi đặt `LIMIT=None` để chạy toàn bộ.

In [15]:
import threading
import queue as _queue

CACHE_PATH = OUT_DIR / "llm_cache.jsonl"
TEMPERATURES = [0.0, 0.3, 0.7]
LIMIT = None                 # <<< None = chạy toàn bộ 1393 câu

# Số luồng cho TỪNG server — đặt theo NĂNG LỰC ĐO ĐƯỢC (benchmark):
#   server1 (phuocnguyn): mạnh ~270 tok/s, bão hoà ở 4 luồng
#   server2 (dutai):      yếu hơn ~150 tok/s, bão hoà ở 2 luồng (lên 4-8 throughput TỤT)
# Dùng HÀNG ĐỢI CHUNG: máy mạnh có nhiều luồng -> tự rút việc nhanh hơn -> cân bằng động.
SERVER_WORKERS = {
    "https://llm.phuocnguyn.id.vn/v1": 4,
    "https://llm2.dutai.site/v1": 2,
}

def load_cache():
    cache = {}
    if CACHE_PATH.exists():
        with open(CACHE_PATH, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                rec = json.loads(line)
                cache[(rec["question_id"], rec["temperature"])] = rec["result"]
    return cache

cache = load_cache()
print("Đã có trong cache:", len(cache), "bản ghi")

work_df = df if LIMIT is None else df.head(LIMIT)
tasks = [(int(r.question_id), t, r) for _, r in work_df.iterrows()
         for t in TEMPERATURES if (int(r.question_id), t) not in cache]
total = len(tasks)
n_threads = sum(SERVER_WORKERS.values())
_layout = ", ".join("{}={}".format(s.split("/")[2], n) for s, n in SERVER_WORKERS.items())
print("Cần gọi LLM: {} lần | {} luồng tổng ({})".format(total, n_threads, _layout))

work_q = _queue.Queue()
for tk in tasks:
    work_q.put(tk)

lock = threading.Lock()
cache_file = open(CACHE_PATH, "a", encoding="utf-8")
progress = {"n": 0}

def worker(server):
    while True:
        try:
            qid, temp, row = work_q.get_nowait()
        except _queue.Empty:
            return
        res = call_llm(build_prompt(row), server=server, temperature=temp)
        with lock:
            cache[(qid, temp)] = res
            if isinstance(res, dict) and "_error" not in res:
                # chỉ ghi kết quả thành công -> lần chạy sau tự thử lại câu lỗi
                cache_file.write(json.dumps(
                    {"question_id": qid, "temperature": temp, "result": res},
                    ensure_ascii=False) + "\n")
                cache_file.flush()
            progress["n"] += 1
            if progress["n"] % 10 == 0 or progress["n"] == total:
                print(f"  {progress['n']}/{total} xong")

threads = []
for server, n in SERVER_WORKERS.items():
    for _ in range(n):
        th = threading.Thread(target=worker, args=(server,), daemon=True)
        th.start()
        threads.append(th)

try:
    for th in threads:
        th.join()
finally:
    cache_file.close()
print("Hoàn tất. Tổng cache:", len(cache))

Đã có trong cache: 4164 bản ghi
Cần gọi LLM: 15 lần | 6 luồng tổng (llm.phuocnguyn.id.vn=4, llm2.dutai.site=2)
  10/15 xong
  15/15 xong
Hoàn tất. Tổng cache: 4179


## Bước 9 — Tổng hợp feature LLM từ cache

Với mỗi câu: số → mean qua 3 temperature; category (ambiguity, bloom) → mode; độ chênh
giữa 3 lần đóng góp vào `H_amb`. Suy ra
`H_N,H_D,H_W,H_amb,H_B,H_M,H_P,H_Dmax,H_Dmean`.

In [10]:
MISC_WEIGHTS = {
    "conceptual": 0.8, "procedural": 0.5, "side-effect trap": 1.0,
    "syntactic trap": 0.6, "lack of sense": 1.2,
}
AMB_MAP = {"low": 0.0, "medium": 0.5, "high": 1.0}

def _num(d, key, default=0.0):
    try:
        v = float(d.get(key))
        return v if math.isfinite(v) else default
    except Exception:
        return default

def aggregate_llm(qid):
    runs = [cache.get((qid, t)) for t in TEMPERATURES]
    runs = [r for r in runs if isinstance(r, dict) and "_error" not in r]
    if not runs:
        return None
    H_N = statistics.mean(_num(r, "step_count") for r in runs)
    H_D = statistics.mean(_num(r, "reasoning_depth") for r in runs)
    W_list = [_num(r, "weighted_steps") for r in runs]
    H_W = statistics.mean(W_list)
    # Bloom: mode
    blooms = [int(_num(r, "bloom_level")) for r in runs if _num(r, "bloom_level") > 0]
    H_B = statistics.mode(blooms) if blooms else 0
    # Ambiguity: trung bình map + cộng độ phân tán của W (chênh lệch lớn => mơ hồ)
    amb_self = statistics.mean(
        AMB_MAP.get(str(r.get("ambiguity", "")).lower(), 0.0) for r in runs)
    w_spread = (statistics.pstdev(W_list) / (statistics.mean(W_list) + 1e-9)
                if len(W_list) > 1 else 0.0)
    H_amb = round(min(1.0, amb_self * 0.7 + min(w_spread, 1.0) * 0.3), 4)
    # Misconceptions: lấy run đầu có danh sách
    miscs = next((r.get("misconceptions") for r in runs
                  if isinstance(r.get("misconceptions"), list) and r["misconceptions"]), [])
    H_M = len(miscs)
    H_P = sum(MISC_WEIGHTS.get(str(m.get("type","")).strip().lower(),
                               _num(m, "weight")) for m in miscs)
    # Distractor plausibility
    scores = []
    for r in runs:
        for d in (r.get("distractor_scores") or []):
            s = _num(d, "score")
            if s > 0:
                scores.append(s)
    H_Dmax = max(scores) if scores else 0.0
    H_Dmean = statistics.mean(scores) if scores else 0.0
    return dict(question_id=qid, H_N=round(H_N,3), H_D=round(H_D,3), H_W=round(H_W,3),
                H_amb=H_amb, H_B=H_B, H_M=H_M, H_P=round(H_P,3),
                H_Dmax=H_Dmax, H_Dmean=round(H_Dmean,3))

llm_rows = [a for qid in df.question_id if (a := aggregate_llm(int(qid))) is not None]
llm_df = pd.DataFrame(llm_rows)
print("Số câu có feature LLM:", len(llm_df))
llm_df.head()

NameError: name 'TEMPERATURES' is not defined

## Bước 10 — Gộp toàn bộ & xuất `features.csv` + `features.json`

In [ ]:
full = df[SURFACE_COLS].merge(llm_df, on="question_id", how="left")

# JSON: giữ cấu trúc lồng (S_ops, T_oop... là list)
full.to_json(OUT_DIR / "features.json", orient="records",
             force_ascii=False, indent=2)
# CSV: phẳng hoá các vector con
explode_vectors(full).to_csv(OUT_DIR / "features.csv", index=False)
print("Đã lưu features.csv / features.json:", full.shape)

# Sanity-check cuối: Q10 phải khó hơn Q7 (H_W lớn hơn) — theo nhận định PDF
cmp_cols = ["question_id","L_lines","S_nest","S_cf","H_W","H_B","H_Dmean"]
print(full[full.question_id.isin([7,10])][cmp_cols].to_string(index=False))
full.describe(include="all").T

Đã lưu features.csv / features.json: (1393, 24)
 question_id  L_lines  S_nest  S_cf   H_W  H_B  H_Dmean
           7        7       1   0.0 6.633  4.0    3.182
          10       14       2   1.5 9.833  4.0    3.143


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
question_id,1393.0,NaN,NaN,NaN,2444.198851,2799.275211,7.0,361.0,2138.0,4118.0,14431.0
L_qtok,1393.0,NaN,NaN,NaN,10.914573,3.776519,0.0,10.0,10.0,11.0,72.0
L_lines,1393.0,NaN,NaN,NaN,17.811199,12.595536,0.0,9.0,15.0,26.0,71.0
L_kw,1393.0,NaN,NaN,NaN,0.276401,0.124032,0.0,0.233333,0.294118,0.354839,0.636364
L_ids,1393.0,NaN,NaN,NaN,5.578607,3.755804,0.0,3.0,5.0,8.0,35.0
S_nest,1393.0,NaN,NaN,NaN,1.102656,0.603105,0.0,1.0,1.0,1.0,4.0
S_cf,1393.0,NaN,NaN,NaN,0.402154,1.005571,0.0,0.0,0.0,0.0,9.0
S_ops,1393,845,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]",251,NaN,NaN,NaN,NaN,NaN,NaN,NaN
T_class,1393.0,NaN,NaN,NaN,0.595118,0.850814,0.0,0.0,0.0,1.0,5.0
T_oop,1393,13,"[0, 0, 0]",807,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Bước 11 — Ghép FULL feature matrix (scalars + embedding 768) cho ML

`features.csv`/`.json` chỉ chứa đặc trưng scalar (dễ đọc). Để huấn luyện, ghép thêm
`E_code` (768) thành ma trận số `feature_matrix.npy` kèm `feature_matrix_meta.json`
(thứ tự `question_id` + tên cột). Cần chạy **Bước 6b** trước.

In [28]:
# Ghép FULL feature matrix = scalar features (đã phẳng hoá) + E_code(768) -> dùng cho ML
import numpy as np

scalar = explode_vectors(df[SURFACE_COLS].merge(llm_df, on="question_id", how="left"))
qid_order = scalar["question_id"].to_numpy()
scalar_cols = [c for c in scalar.columns if c != "question_id"]
X_scalar = scalar[scalar_cols].to_numpy(dtype="float32")   # có thể có NaN ở câu thiếu LLM

E = np.load(OUT_DIR / "code_embeddings.npy")               # [N, 768], cùng thứ tự df
assert len(E) == len(X_scalar), "Số dòng embedding != số câu — chạy lại Bước 6b"

X_full = np.hstack([X_scalar, E]).astype("float32")
emb_cols = [f"E_code_{i}" for i in range(E.shape[1])]
all_cols = scalar_cols + emb_cols

np.save(OUT_DIR / "feature_matrix.npy", X_full)
with open(OUT_DIR / "feature_matrix_meta.json", "w", encoding="utf-8") as f:
    json.dump({"question_id_order": qid_order.tolist(), "columns": all_cols},
              f, ensure_ascii=False)
print("Đã lưu feature_matrix.npy:", X_full.shape,
      f"({len(scalar_cols)} scalar + {len(emb_cols)} embedding)")
print("Kèm feature_matrix_meta.json (thứ tự question_id + tên cột)")

Đã lưu feature_matrix.npy: (1393, 803) (35 scalar + 768 embedding)
Kèm feature_matrix_meta.json (thứ tự question_id + tên cột)


In [ ]:
uv run python scripts/load_features.py

Note: you may need to restart the kernel to use updated packages.


c:\Users\Admin\AppData\Local\Programs\Python\Python313\python.exe: No module named uv
